# 이차전지 데이터 분석 부트캠프 중급 · 학부 Final Mission

## 새로운 배터리의 SOH 예측 문제를 해결하라

---

### 이 노트북 사용법

- **코드는 직접 작성합니다.** 아래는 무엇을 할지와 어떤 함수를 쓰는지만 알려줍니다.
- 막히면 **2일차 대조표**(Orange3 ↔ 파이썬)를 보십시오.
- AI에게 물어봐도 됩니다. 다만 **내 생각을 먼저 제시**하고 검토받으십시오.
- 제출 전 **런타임 → 모두 실행**으로 처음부터 끝까지 도는지 확인하십시오.

### 이름을 적으십시오



In [ ]:
이름 = ""        # 예: "홍길동"
학과 = ""
print(f"{이름} · {학과}")


---
## 0. 준비

필요한 라이브러리를 불러오고 데이터를 읽습니다.

**힌트**
- `pandas` `numpy` `matplotlib.pyplot`
- sklearn에서 필요한 것: 모델 · `cross_validate` · `KFold` · 지표
- 데이터 업로드: 왼쪽 파일 아이콘 → 업로드, 또는 `files.upload()`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# TODO — sklearn에서 필요한 것을 불러오십시오




In [ ]:
# battery_mission.csv 를 읽으십시오
df = 

# 크기와 앞 5행을 확인하십시오




---
## 1. 실험 환경 확인 — 분석 전에

**숫자는 실험 조건을 모르면 읽을 수 없습니다.**

### 사양서에서 확인한 것 (실험 사양서 문서 참조)

| 항목 | 내용 |
|---|---|
| 기록 구간 | **방전만** — 충전 정보 없음 |
| 온도 | 43℃ · 4℃ 두 종류 |
| 컷오프 | 2.0 · 2.2 · 2.5 · 2.7 V — **두 온도에 동일 구성** |
| 초기 용량 | 43℃ 약 2.00 Ah · 4℃ 약 1.23 Ah |
| 회복 현상 | 실제 현상 — **이상치가 아님** |
| 조건당 셀 | **1개** — 개체차와 구분 불가 |

### 직접 조사할 것 (발표에 기재)

1. SOH의 정의 — 무엇을 100%로 잡았는가
2. 종료 조건 — 왜 그 사이클에서 멈췄는가
3. 컷오프 설계 — 왜 셀마다 다른가
4. 측정 주기 — 임피던스를 몇 사이클마다 쟀는가

> **nasa.gov → PCoE Data Set Repository → "5. Batteries"**


In [ ]:
조사_결과 = {
    "SOH 정의": "",
    "종료 조건": "",
    "컷오프 설계": "",
    "측정 주기": "",
    "Citation": "",
}
for k, v in 조사_결과.items():
    print(f"{k}: {v or '(확인하지 못함)'}")


### 사양이 맞는지 데이터로 확인

문서에서 읽은 것과 데이터가 일치하는지 대조하십시오.

**힌트** — 온도별 초기 용량 · 컷오프 구성 · 결측 위치


In [ ]:
# 셀별 조건표를 만들어 사양과 대조하십시오
# 힌트: groupby('cell_id').agg(...) 로 온도·컷오프·초기용량·사이클수




---
## 2. 데이터 확인 — GATE 1

### 확인할 것
- 몇 개 셀 · 몇 사이클인가
- `ambient_temp` 가 몇 종류인가 — **섞여 있습니다**
- 결측이 어디에 몰려 있는가 · 구조적인가
- SOH 범위는 얼마인가

**힌트** — `.shape` `.info()` `.describe()` `.isna().sum()` `.value_counts()` `.groupby()`


In [ ]:
# 크기 · 컬럼 · 자료형




In [ ]:
# 셀별 · 온도별 요약
# 힌트: groupby(['ambient_temp','cell_id']).agg(...)




In [ ]:
# 결측 위치 — 어느 컬럼에 몇 개인가, 어느 사이클에 몰려 있는가




### 누수 점검

SOH와 지나치게 상관이 높은 변수가 있는지 확인하십시오.

> **판단 기준** — 이 값을 **운행 중에 알 수 있는가?**
>
> 알 수 없다면 누수이거나, 최소한 쓸 수 없는 변수입니다.

**힌트** — `df.corr()['soh'].sort_values(key=abs, ascending=False)`


In [ ]:
# 상관 순위




### ★ 여기서 한 번 멈추십시오

전체로 본 상관과 **온도 그룹별로 본 상관**이 다를 수 있습니다.
두 가지를 모두 확인하고, **어느 쪽이 맞는지** 판단하십시오.

이것이 **GATE 3(과도한 해석)** 을 시험하는 지점입니다.


In [ ]:
# 온도 그룹별 상관
# 힌트: for t in df.ambient_temp.unique(): ...




---
## 3. 시각화

그래프는 **질문에 대한 답변 양식**입니다. 무엇을 묻는지 먼저 정하십시오.

**힌트**
- 열화 곡선: `plt.plot(cycle, soh)` 을 셀별로 반복
- 온도 그룹 비교: 색을 나눠 겹쳐 그리기
- 한글은 깨지므로 **라벨은 영문**으로
- 기준선: `plt.axhline(80, ls='--', c='red')`


In [ ]:
# 셀별 열화 곡선




In [ ]:
# 내 질문에 필요한 그래프




---
## 4. Baseline — 기준을 먼저 만듭니다

**개선을 말하려면 비교 대상이 있어야 합니다.**
가장 단순한 형태로 하나 만들고 성능을 기록하십시오.

### 정할 것
1. 입력 변수 — 무엇을 쓸 것인가 (누수는 뺐습니까?)
2. 평가 방식 — 무작위 · 시간순 · 셀 단위 중 무엇인가 · **왜인가** (GATE 2)
3. 모델 — 단순한 것부터

**힌트** — `cross_validate(model, X, y, cv=cv, scoring=('r2','neg_mean_absolute_error'))`


In [ ]:
# 입력 변수 목록
F = [

]

# 결측 처리 후 X, y




In [ ]:
# 평가 방식 — 왜 이것을 골랐는지 워크시트에 적으십시오
cv = 

# Baseline 모델




In [ ]:
# Baseline 성능 — 이 수치를 기록하십시오




> **Baseline 성능을 워크시트 RESULT 칸에 적으십시오.**
> 이후 모든 실험을 이 수치와 비교합니다.


---
## 5. AI Review — 가설을 검토받습니다

### 순서를 지키십시오

1. 데이터를 보고 **내가 먼저** 개선 가설을 세운다
2. 그 생각과 근거를 AI에게 **제시한다**
3. 반론·대안을 받는다
4. **내가** 선택·수정한다

### 이렇게 물으십시오

```
온도 그룹이 두 개라 초기 용량이 다르다고 봤어.
그래서 그룹을 나눠 학습하려는데, 데이터가 반으로 줄어 걱정돼.
이 판단에 대한 반론을 알려줘.
```

**워크시트 3번 칸(AI REVIEW)에 받은 지적을 그대로 적으십시오.**


In [ ]:
# AI 지적을 데이터로 확인해 보십시오
# 예 — 그룹을 나누면 각각 몇 행이 되는가




---
## 6. 실험 — 개선하고 비교합니다

한 번에 하나씩 바꾸십시오. **무엇 때문에 달라졌는지** 알 수 없게 됩니다.

| No | 바꾼 것 | 결과 | Baseline 대비 |
|---|---|---|---|
| 0 | Baseline | | — |
| 1 | | | |
| 2 | | | |


In [ ]:
# 실험 1




In [ ]:
# 실험 2




In [ ]:
# 실험 결과를 표로 정리하십시오
# 힌트: 결과를 dict 리스트로 모아 pd.DataFrame() 으로




---
## 7. 해석 — 내가 먼저

결과를 보고 **왜 이렇게 되었는지** 먼저 설명해 보십시오.

### 확인할 것 (GATE 3)
- 성능이 올랐다면 **어떤 평가에서** 올랐는가
- 다른 평가 방식으로도 확인해 봤는가
- 상관이 높은 것을 **원인으로 단정**하지 않았는가


In [ ]:
# 다른 평가 방식으로도 확인




---
## 8. AI Critique — 해석을 검토받습니다

```
그룹을 나눠 학습했더니 R²가 0.02 올랐어.
그래서 그룹 분리가 효과적이라고 결론냈는데,
내 해석에서 과도한 부분이나 다른 설명이 가능한 지점을 짚어 줘.
```

**워크시트 8번 칸에 적고, 지적받은 것을 아래에서 확인하십시오.**


In [ ]:
# AI 지적을 데이터로 확인




---
## 마무리 — 한계와 제출 확인

### 한계 (GATE 4)

이 결론은 **어떤 조건에서만** 유효합니까?

- 데이터 크기 — 몇 셀 · 몇 사이클인가
- 실험 조건 — 어떤 온도 · 어떤 프로파일인가
- 확인하지 못한 것 — 무엇을 알 수 없는가
- 필요한 것 — 무엇이 있어야 답할 수 있는가


In [ ]:
한계 = [
    "",
    "",
    "",
]
for i, x in enumerate(한계, 1):
    print(f"{i}. {x}")


---
### 제출 전 확인

- [ ] **런타임 → 모두 실행**으로 처음부터 끝까지 오류 없이 돌아간다
- [ ] 워크시트 9칸을 모두 채웠다
- [ ] AI에게 **내 생각을 먼저** 제시했고 그 기록이 있다
- [ ] GATE 1~4를 스스로 점검했다
- [ ] 결론에 조건이 붙어 있다

### 파일 이름

`미션_이름.ipynb` 으로 저장해 제출하십시오.

---

인공지능융합교육원 주식회사 · 이차전지 데이터 분석 부트캠프 중급
